# Evaluation: Ensemble Teacher & DUDES Student

Inference of the **Ensemble** (3 fair folds) and **DUDES** on the held-out test split for each `(arch × test year)` and collects per-image arrays for downstream analysis.

**Fair Ensemble Mapping**

| Test Year | Ensemble Folds |
|-----------|----------------|
| 2018      | 7, 9, 11       |
| 2019      | 3, 5, 10       |
| 2020      | 1, 4, 8        |
| 2021      | 0, 2, 6        |

**Cells:**
1. Configuration 
2. Helper Functions
3. Collect Per-Image Predictions
4. Per-Year Average Precision (AP)
5. Average Surface Distance (ASD)
6. FCER sweep: AUROC, AUPRC, Brier, NLL
7. AUPRC Random Baseline
8. Wilcoxon Signed-Rank Test
9. Plot: FCER Sweep
10. Plot: Qualitative Figure


### 1. Configuration

In [ ]:
from pathlib import Path

BASE_DIR      = Path().resolve().parent
_WEIGHTS_ROOT = BASE_DIR / "pretrained_weights"

DATA_DIR   = BASE_DIR / "WildfireSpreadTS_HDF5"
LOAD_HDF5  = True

OUTPUT_DIR         = BASE_DIR / "results_test" / "Evaluation_UTAE_T5"    # eval results saved here
DUDES_TRAINING_DIR = BASE_DIR / "results" / "DUDES_Training_UTAE_T5"  # trained head checkpoints
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Architecture registry ──────────────────────────────────────
ARCH_CONFIG = {
    "utae_t5": {
        "weights_dir":      _WEIGHTS_ROOT / "Res18UTAE_T5" / "Veg",
        "n_timesteps":      5,
        "model_class":      "SMPTempModel",
        "flatten_temporal": False,
    },
    "unet_t5": {
        "weights_dir":      _WEIGHTS_ROOT / "Res18Unet_T5" / "Veg",
        "n_timesteps":      5,
        "model_class":      "SMPModel",
        "flatten_temporal": True,
    },
    "unet_t1": {
        "weights_dir":      _WEIGHTS_ROOT / "Res18Unet_T1" / "Veg",
        "n_timesteps":      1,
        "model_class":      "SMPModel",
        "flatten_temporal": True,
    },
}

#ARCHS    = ["utae_t5", "unet_t5", "unet_t1"]
ARCHS    = ["utae_t5"]
FEATURES = [0, 1, 2, 3, 4, 38, 39]

# ── Fair ensemble mapping ──────────────────────────────────────
YEAR_TO_FOLDS = {
    2018: [7, 9, 11],
    2019: [3, 5, 10],
    2020: [1, 4, 8],
    2021: [0, 2, 6],
}
TEST_YEARS = [2018, 2019, 2020, 2021]


# ── Backbone fold selection ──
# "best" | "middle" | "worst"  — selects backbone from the 3 fair folds by testAP rank.
# "middle" (median) is robust and avoids selection bias.
BB_FOLD_SELECTION = "middle"

BATCH_SIZE   = 4
NUM_WORKERS  = 0

print(f"Architectures    : {ARCHS}")
print(f"Test years       : {TEST_YEARS}")
print(f"Data dir         : {DATA_DIR}")
print(f"Output dir       : {OUTPUT_DIR}")
print(f"DUDES ckpt dir   : {DUDES_TRAINING_DIR}")


### 2. Helper Functions

In [ ]:
import re
import sys
import types

import numpy as np
import torch
import torch.nn as nn

sys.modules.setdefault("wandb", types.ModuleType("wandb"))

for _p in [str(BASE_DIR / "third_party"), str(BASE_DIR / "src")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from models.SMPModel     import SMPModel
from models.SMPTempModel import SMPTempModel
from dataloader.FireSpreadDataModule import FireSpreadDataModule

device = torch.device("cpu")
print(f"Device: {device}")


# ── Parse testAP from checkpoint filename ──────────────────────
def _parse_ap(p):
    m = re.search(r"testAP(\d+\.\d+)", p.name)
    return float(m.group(1)) if m else -1.0


def resolve_bb_fold(pth_paths, year):
    """Return (fold_id, testAP) for the backbone fold of the given test year."""
    fold_ids = YEAR_TO_FOLDS[year]
    ranked   = sorted(fold_ids, key=lambda fid: _parse_ap(pth_paths[fid]))
    if BB_FOLD_SELECTION == "best":
        fid = ranked[-1]
    elif BB_FOLD_SELECTION == "worst":
        fid = ranked[0]
    else:
        fid = ranked[len(ranked) // 2]
    return fid, _parse_ap(pth_paths[fid])


def load_backbone(arch: str, pth_path) -> nn.Module:
    """Load a pretrained backbone (SMPTempModel or SMPModel) from a .pth file."""
    cfg   = ARCH_CONFIG[arch]
    state = torch.load(str(pth_path), map_location=device, weights_only=True)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    n_ch = next(v.shape[1] for k, v in state.items()
                if k.endswith("encoder.conv1.weight"))
    if cfg["model_class"] == "SMPTempModel":
        model = SMPTempModel(
            encoder_name="resnet18", n_channels=n_ch,
            flatten_temporal_dimension=cfg["flatten_temporal"],
            pos_class_weight=10.0, encoder_weights=None, loss_function="BCE",
        )
    else:
        model = SMPModel(
            encoder_name="resnet18", n_channels=n_ch,
            flatten_temporal_dimension=cfg["flatten_temporal"],
            pos_class_weight=10.0, encoder_weights=None, loss_function="BCE",
        )
    nn.Module.load_state_dict(model, state, strict=False)
    return model.eval().to(device)


# ── DUDES model wrapper ────────────────────────────────────────
class ModelWithUncertaintyHead(nn.Module):
    """Backbone + 1×1 conv uncertainty head. Matches 01_train_dudes.ipynb."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        dec_out_ch    = backbone.model.segmentation_head[0].in_channels
        self.uncertainty_head = nn.Sequential(
            nn.Conv2d(dec_out_ch, 1, kernel_size=1),
            nn.Sigmoid(),
        )
        self._dec = None
        backbone.model.decoder.register_forward_hook(
            lambda m, i, o: setattr(self, "_dec", o)
        )

    def forward(self, x):
        seg_logit = self.backbone(x)
        unc       = self.uncertainty_head(self._dec)
        return seg_logit, unc

# Maximum possible sample std for N=3 sigmoid outputs in [0, 1].
# Achieved at configurations {0, 0, 1} or {0, 1, 1}:
#   s = sqrt( k*(N-k) / (N*(N-1)) )  with k=1, N=3  →  sqrt(1/3) = 1/sqrt(3) ≈ 0.5774
# Dividing by this maps the teacher's full std range into [0, 1] without premature clamping.
_TEACHER_STD_MAX = (1.0 / 3.0) ** 0.5  # = 1/sqrt(3) ≈ 0.5774

def ensemble_uncertainty(logit_list):
    """Normalised std of sigmoid predictions across ensemble members → [0, 1]."""
    probs = torch.stack([torch.sigmoid(l) for l in logit_list], dim=0)
    return (probs.std(dim=0) / _TEACHER_STD_MAX).clamp(0.0, 1.0)


print("Helpers ready.")
print(f"_TEACHER_STD_MAX = {_TEACHER_STD_MAX:.4f}")

### 3. Collect Per-Image Predictions

Runs inference for every `(arch × year)` combination and stores four arrays per image:

| Variable | Shape | Description |
|---|---|---|
| `unc_ensemble` | H × W | Normalised std of 3 sigmoid outputs (teacher uncertainty) |
| `unc_dudes` | H × W | DUDES head output (student uncertainty) |
| `prob` | H × W | Sigmoid of the DUDES backbone segmentation logit |
| `y` | H × W | Ground-truth fire label |

Results are written to `OUTPUT_DIR/eval_{arch}_year{year}.npz`. Already-existing files are skipped.


In [ ]:
for ARCH in ARCHS:
    cfg    = ARCH_CONFIG[ARCH]
    n_obs  = cfg["n_timesteps"]

    pth_paths = sorted(
        cfg["weights_dir"].glob("fold*.pth"),
        key=lambda p: int(p.stem.split("_")[0].replace("fold", "")),
    )

    print(f"\n{'#'*64}")
    print(f"  Evaluation — {ARCH}")
    print(f"{'#'*64}")

    for YEAR in TEST_YEARS:
        out_path = OUTPUT_DIR / f"eval_{ARCH}_year{YEAR}.npz"
        if out_path.exists():
            print(f"\n  {YEAR}: results already exist ({out_path.name}), skipping.")
            continue

        bb_fold, bb_ap = resolve_bb_fold(pth_paths, YEAR)
        ens_folds      = YEAR_TO_FOLDS[YEAR]
        ckpt_dir       = DUDES_TRAINING_DIR / f"{ARCH}_year{YEAR}"

        # ── Find best DUDES head checkpoint ──────────────────
        def _vauroc(p):
            m = re.search(r"vauroc=(\d+\.\d+)", p.name)
            return float(m.group(1)) if m else -1.0

        head_ckpts = sorted(ckpt_dir.glob("head_*.pt"), key=_vauroc, reverse=True) if ckpt_dir.exists() else []
        assert head_ckpts, f"No head_*.pt found in {ckpt_dir}. Run 01_train_dudes.ipynb first."
        head_ckpt_path = head_ckpts[0]   # highest val_unc_auroc

        print(f"\n  {ARCH}  test year {YEAR}")
        print(f"    Ensemble folds : {ens_folds}")
        print(f"    Backbone fold  : {bb_fold}  (testAP={bb_ap:.3f})")
        print(f"    DUDES ckpt     : {head_ckpt_path.name}")

        # ── Load ensemble teachers ────────────────────────────
        print("  Loading ensemble teachers...")
        teachers = [load_backbone(ARCH, pth_paths[f]) for f in ens_folds]

        # ── Load DUDES student ────────────────────────────────
        print("  Loading DUDES student...")
        head_ckpt = torch.load(str(head_ckpt_path), map_location=device)
        backbone  = load_backbone(ARCH, pth_paths[bb_fold])
        dudes     = ModelWithUncertaintyHead(backbone).eval()
        dudes.uncertainty_head.load_state_dict(head_ckpt["uncertainty_head"])

        # ── Test data loader for this year ────────────────────
        dm = FireSpreadDataModule(
            data_dir=str(DATA_DIR), batch_size=1,
            n_leading_observations=n_obs, crop_side_length=128,
            load_from_hdf5=LOAD_HDF5, num_workers=NUM_WORKERS,
            remove_duplicate_features=False, features_to_keep=FEATURES,
            n_leading_observations_test_adjustment=n_obs,
            data_fold_id=bb_fold, return_doy=False,
        )
        dm.setup("test")
        test_loader = dm.test_dataloader()
        print(f"  Test samples    : {len(test_loader.dataset)}")

        # ── Inference loop ────────────────────────────────────
        all_unc_ens   = []
        all_prob_ens  = []
        all_unc_dudes = []
        all_prob      = []
        all_y         = []

        with torch.no_grad():
            for i, batch in enumerate(test_loader):
                x = batch[0].to(device)
                y = batch[1].float().squeeze(0).squeeze(0)   # (H, W)
                # Teacher: ensemble std + mean prob
                logits   = [t(x).squeeze(1) for t in teachers]
                unc_ens  = ensemble_uncertainty(logits).squeeze(0)   # (H, W)
                prob_ens = torch.stack([torch.sigmoid(l) for l in logits], dim=0).mean(dim=0).squeeze(0)  # (H, W)

                # Backbone prob + DUDES uncertainty
                seg_logit, unc_t = dudes(x)
                prob      = torch.sigmoid(seg_logit).squeeze(0).squeeze(0)  # (H, W)
                unc_dudes = unc_t.squeeze(0).squeeze(0)                     # (H, W)

                all_prob_ens.append(prob_ens.numpy().astype(np.float32))
                all_unc_dudes.append(unc_dudes.numpy().astype(np.float32))
                all_prob.append(prob.numpy().astype(np.float32))
                all_y.append(y.numpy().astype(np.float32))
                all_unc_ens.append(unc_ens.numpy().astype(np.float32))

                if (i + 1) % 100 == 0:
                    print(f"    {i+1}/{len(test_loader)}")

        # ── Save results ──────────────────────────────────────
        np.savez_compressed(
            out_path,
            unc_ensemble  = np.stack(all_unc_ens),
            prob_ensemble = np.stack(all_prob_ens),
            unc_dudes     = np.stack(all_unc_dudes),
            prob          = np.stack(all_prob),
            y             = np.stack(all_y),
        )
        n = len(all_y)
        n_fire = sum(yi.sum() > 0 for yi in all_y)
        print(f"  Saved → {out_path.name}")
        print(f"    samples={n}  fire={n_fire}  no-fire={n - n_fire}")

        del teachers, dudes, backbone, dm

print(f"\n{'#'*64}")
print("  Done.")
print(f"{'#'*64}")


### 4. Average Precision (AP)

- **Backbone/DUDES AP**: computed from the single backbone sigmoid (`prob`)
- **Ensemble AP**: computed from the mean sigmoid across 3 ensemble members (`prob_ensemble`)


In [ ]:
from sklearn.metrics import average_precision_score

_C1, _C2, _C3, _C4 = 6, 18, 14, 12   # display column widths

for ARCH in ARCHS:
    print(f"\n{'#'*64}")
    print(f"  AP Results — {ARCH}")
    print(f"{'#'*64}")
    print(f"  {'Year':<{_C1}}  {'Backbone/DUDES AP':>{_C2}}  {'Ensemble AP':>{_C3}}  {'#pixels':>{_C4}}")
    print(f"  {'-'*_C1}  {'-'*_C2}  {'-'*_C3}  {'-'*_C4}")

    ap_bb_vals  = []
    ap_ens_vals = []

    for YEAR in TEST_YEARS:
        npz_path = OUTPUT_DIR / f"eval_{ARCH}_year{YEAR}.npz"
        if not npz_path.exists():
            print(f"  {YEAR:<{_C1}}  {'[missing]':>{_C2}}")
            continue

        data = np.load(str(npz_path), allow_pickle=True)

        y_flat    = np.concatenate([arr.ravel() for arr in data["y"]])
        prob_flat = np.concatenate([arr.ravel() for arr in data["prob"]])           # backbone sigmoid
        ens_flat  = np.concatenate([arr.ravel() for arr in data["prob_ensemble"]])  # ensemble mean sigmoid

        # Drop non-binary pixels (NaN / ignore-class)
        valid     = (y_flat == 0) | (y_flat == 1)
        y_flat    = y_flat[valid].astype(np.int32)
        prob_flat = prob_flat[valid]
        ens_flat  = ens_flat[valid]

        if y_flat.sum() == 0:
            print(f"  {YEAR:<{_C1}}  {'[no fire pixels]':>{_C2}}")
            continue

        ap_bb  = average_precision_score(y_flat, prob_flat)
        ap_ens = average_precision_score(y_flat, ens_flat)

        ap_bb_vals.append(ap_bb)
        ap_ens_vals.append(ap_ens)

        print(f"  {YEAR:<{_C1}}  {ap_bb:>{_C2}.4f}  {ap_ens:>{_C3}.4f}  {len(y_flat):>{_C4},}")

    # ── Mean ± std across years ───────────────────────────────
    if ap_bb_vals:
        bb_mean,  bb_std  = np.mean(ap_bb_vals),  np.std(ap_bb_vals)
        ens_mean, ens_std = np.mean(ap_ens_vals), np.std(ap_ens_vals)
        bb_str  = f"{bb_mean:.4f}±{bb_std:.4f}"
        ens_str = f"{ens_mean:.4f}±{ens_std:.4f}"
        print(f"\n  {'─'*(_C1 + 2 + _C2 + 2 + _C3 + 2 + _C4)}")
        print(f"  {'Mean±Std':<{_C1}}  {bb_str:>{_C2}}  {ens_str:>{_C3}}")


###  5. Average Surface Distance (ASD)

Computed in metres between the predicted fire boundary and the ground-truth fire boundary. Pixel resolution is 375 m (VIIRS).

- **Backbone/DUDES ASD** — predicted mask: `prob > 0.5`
- **Ensemble ASD** — predicted mask: `prob_ensemble > 0.5`

Only fire images (at least one GT fire pixel) are included.


In [ ]:
from scipy.spatial.distance import cdist
from scipy.ndimage import binary_dilation

PIXEL_RES_M = 375.0   # VIIRS resolution in metres


def compute_asd(pred_mask, gt_mask, pixel_res_m=PIXEL_RES_M):
    """Symmetric Average Surface Distance in metres (disk boundary)."""
    def _boundary(mask):
        m      = mask.astype(bool)
        r      = 1
        _y, _x = np.ogrid[-r:r+1, -r:r+1]
        disk   = (_y**2 + _x**2) <= r**2
        return np.argwhere(binary_dilation(m, structure=disk) & ~m).astype(np.float64) * pixel_res_m

    pp, gp = _boundary(pred_mask), _boundary(gt_mask)
    if not len(pp) or not len(gp):
        return np.nan
    D = cdist(pp, gp)
    return float((D.min(axis=1).mean() + D.min(axis=0).mean()) / 2.0)


_C1, _C2, _C3, _C4, _C5, _C6 = 6, 18, 14, 6, 8, 9   # column widths

for ARCH in ARCHS:

    asd_bb_means  = []
    asd_ens_means = []
    rows = []

    for YEAR in TEST_YEARS:
        npz_path = OUTPUT_DIR / f"eval_{ARCH}_year{YEAR}.npz"
        if not npz_path.exists():
            rows.append((YEAR, np.nan, np.nan, 0, 0, 0))
            continue

        data     = np.load(str(npz_path), allow_pickle=True)
        probs    = data["prob"]
        prob_ens = data["prob_ensemble"]
        ys       = data["y"]

        asd_bb_vals  = []
        asd_ens_vals = []
        n_fire = n_miss_bb = n_miss_ens = 0

        for prob_i, prob_ens_i, y_i in zip(probs, prob_ens, ys):
            gt = y_i.astype(bool)
            if not gt.any():
                continue
            n_fire += 1

            pred_bb  = (prob_i     > 0.5)
            pred_ens = (prob_ens_i > 0.5)

            if not pred_bb.any():
                n_miss_bb += 1
            else:
                v = compute_asd(pred_bb, gt)
                if not np.isnan(v):
                    asd_bb_vals.append(v)

            if not pred_ens.any():
                n_miss_ens += 1
            else:
                v = compute_asd(pred_ens, gt)
                if not np.isnan(v):
                    asd_ens_vals.append(v)

        mean_bb  = np.mean(asd_bb_vals)  if asd_bb_vals  else np.nan
        mean_ens = np.mean(asd_ens_vals) if asd_ens_vals else np.nan

        asd_bb_means.append(mean_bb)
        asd_ens_means.append(mean_ens)

        rows.append((YEAR, mean_bb, mean_ens, n_fire, n_miss_bb, n_miss_ens))

    # ── Table: MEAN ASD ───────────────────────────────────────
    print(f"\n{'#'*64}")
    print(f"  ASD — {ARCH}  [metres, disk boundary, 375m/px]")
    print(f"{'#'*64}")
    print(f"  {'Year':<{_C1}}  {'Backbone/DUDES':>{_C2}}  {'Ensemble':>{_C3}}  "
          f"{'#fire':>{_C4}}  {'#miss_bb':>{_C5}}  {'#miss_ens':>{_C6}}")
    print(f"  {'-'*_C1}  {'-'*_C2}  {'-'*_C3}  {'-'*_C4}  {'-'*_C5}  {'-'*_C6}")

    for YEAR, mean_bb, mean_ens, n_fire, n_miss_bb, n_miss_ens in rows:
        bb_s  = f"{mean_bb:.1f}m"  if not np.isnan(mean_bb)  else "[missing]"
        ens_s = f"{mean_ens:.1f}m" if not np.isnan(mean_ens) else "[missing]"
        print(f"  {YEAR:<{_C1}}  {bb_s:>{_C2}}  {ens_s:>{_C3}}  "
              f"{n_fire:>{_C4}}  {n_miss_bb:>{_C5}}  {n_miss_ens:>{_C6}}")

    # ── Mean ± std across years ───────────────────────────────
    valid_bb  = [v for v in asd_bb_means  if not np.isnan(v)]
    valid_ens = [v for v in asd_ens_means if not np.isnan(v)]
    bb_m,  bb_s  = (np.mean(valid_bb),  np.std(valid_bb))  if valid_bb  else (np.nan, np.nan)
    ens_m, ens_s = (np.mean(valid_ens), np.std(valid_ens)) if valid_ens else (np.nan, np.nan)
    bb_str  = f"{bb_m:.1f}±{bb_s:.1f}m"
    ens_str = f"{ens_m:.1f}±{ens_s:.1f}m"
    print(f"\n  {'─'*(_C1 + 2 + _C2 + 2 + _C3 + 2 + _C4 + 2 + _C5 + 2 + _C6)}")
    print(f"  {'Mean±Std':<{_C1}}  {bb_str:>{_C2}}  {ens_str:>{_C3}}")


### 6. FCER sweep: AUROC, AUPRC, Brier, NLL

Evaluates uncertainty quality over FCER with expanding circular dilation radius ($r_d$ = 0–10 px, 1 px steps) around the GT fire mask.


In [ ]:
from scipy.ndimage import binary_dilation
from sklearn.metrics import roc_auc_score, average_precision_score
import time

MAX_RADIUS_PX = 10
BUFFER_RADII  = list(range(0, MAX_RADIUS_PX + 1)) + ["full"]

_H1, _H2, _H3, _H4, _H5 = 10, 10, 10, 10, 10   # column widths for metric values


def _disk(r):
    """Boolean disk structuring element of integer radius r (pixels)."""
    if r == 0:
        s = np.zeros((3, 3), dtype=bool); s[1, 1] = True
        return s
    _y, _x = np.ogrid[-r:r+1, -r:r+1]
    return (_y**2 + _x**2) <= r**2


def _metrics(unc_vals, prob_vals, y_vals, ref_prob_vals=None):
    """AUROC, AUPRC (unc→error), Brier, NLL on flat arrays.

    ref_prob_vals: if provided, the binary error map for AUROC/AUPRC is derived
                   from ref_prob_vals instead of prob_vals.  Use this to share
                   the backbone error map across both methods so that AUROC/AUPRC
                   answer the same question (backbone mistakes) for a fair comparison.
                   Brier and NLL always use prob_vals (each method's own probabilities).
    """
    y     = y_vals.astype(np.int32)
    p     = prob_vals.clip(1e-7, 1 - 1e-7)
    p_ref = ref_prob_vals.clip(1e-7, 1 - 1e-7) if ref_prob_vals is not None else p
    err   = ((p_ref > 0.5).astype(np.int32) != y).astype(np.int32)
    auroc = roc_auc_score(err, unc_vals)           if (err.sum() > 0 and (1 - err).sum() > 0) else np.nan
    auprc = average_precision_score(err, unc_vals) if err.sum() > 0                            else np.nan
    brier = float(np.mean((p - y) ** 2))
    nll   = float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
    return auroc, auprc, brier, nll


sweep_results = {}   # sweep_results[ARCH][YEAR]["dudes"|"ensemble"]["auroc"|"auprc"|"brier"|"nll"]

for ARCH in ARCHS:
    sweep_results[ARCH] = {}

    print(f"\n{'#'*80}")
    print(f"  Buffer sweep (r=0–{MAX_RADIUS_PX}px + full) — {ARCH}")
    print(f"{'#'*80}")

    for YEAR in TEST_YEARS:
        npz_path = OUTPUT_DIR / f"eval_{ARCH}_year{YEAR}.npz"
        if not npz_path.exists():
            print(f"  {YEAR}: [missing npz], skipping.")
            continue

        data      = np.load(str(npz_path), allow_pickle=True)
        probs     = data["prob"]
        prob_ens  = data["prob_ensemble"]
        unc_dudes = data["unc_dudes"]
        unc_ens   = data["unc_ensemble"]
        ys        = data["y"]

        fire_idx = [i for i, y_i in enumerate(ys) if y_i.astype(bool).any()]
        print(f"\n  {YEAR}  fire images={len(fire_idx)}/{len(ys)}")

        # header
        print(f"  {'r':<8}  "
              f"{'AUROC_dud':>{_H1}}  {'AUROC_ens':>{_H2}}  "
              f"{'AUPRC_dud':>{_H3}}  {'AUPRC_ens':>{_H4}}  "
              f"{'Brier_dud':>{_H5}}  {'Brier_ens':>{_H5}}  "
              f"{'NLL_dud':>{_H5}}  {'NLL_ens':>{_H5}}")
        print(f"  {'-'*8}  {'-'*_H1}  {'-'*_H2}  {'-'*_H3}  {'-'*_H4}  {'-'*_H5}  {'-'*_H5}  {'-'*_H5}  {'-'*_H5}")

        res = {m: {met: [] for met in ["auroc", "auprc", "brier", "nll"]}
               for m in ["dudes", "ensemble"]}

        t0 = time.time()
        for step, r in enumerate(BUFFER_RADII):
            structure = None if r == "full" else _disk(r)

            bb_unc_all, bb_prob_all, y_bb_all    = [], [], []
            ens_unc_all, ens_prob_all, y_ens_all = [], [], []

            for i in fire_idx:
                gt   = ys[i].astype(bool)
                mask = np.ones((128, 128), dtype=bool) if r == "full" else binary_dilation(gt, structure=structure)
                flat = mask.ravel()

                bb_unc_all.append(unc_dudes[i].ravel()[flat])
                bb_prob_all.append(probs[i].ravel()[flat])
                y_bb_all.append(ys[i].ravel()[flat])

                ens_unc_all.append(unc_ens[i].ravel()[flat])
                ens_prob_all.append(prob_ens[i].ravel()[flat])
                y_ens_all.append(ys[i].ravel()[flat])

            auroc_bb,  auprc_bb,  brier_bb,  nll_bb  = _metrics(
                np.concatenate(bb_unc_all), np.concatenate(bb_prob_all), np.concatenate(y_bb_all))
            auroc_ens, auprc_ens, brier_ens, nll_ens = _metrics(
                np.concatenate(ens_unc_all), np.concatenate(ens_prob_all), np.concatenate(y_ens_all),
                ref_prob_vals=np.concatenate(bb_prob_all))  # shared backbone error map

            for key, val in zip(["auroc", "auprc", "brier", "nll"],
                                 [auroc_bb, auprc_bb, brier_bb, nll_bb]):
                res["dudes"][key].append(val)
            for key, val in zip(["auroc", "auprc", "brier", "nll"],
                                 [auroc_ens, auprc_ens, brier_ens, nll_ens]):
                res["ensemble"][key].append(val)

            r_label = f"r={r}px" if r != "full" else "full"
            print(f"  {r_label:<8}  "
                  f"{auroc_bb:>{_H1}.4f}  {auroc_ens:>{_H2}.4f}  "
                  f"{auprc_bb:>{_H3}.4f}  {auprc_ens:>{_H4}.4f}  "
                  f"{brier_bb:>{_H5}.4f}  {brier_ens:>{_H5}.4f}  "
                  f"{nll_bb:>{_H5}.4f}  {nll_ens:>{_H5}.4f}")

        sweep_results[ARCH][YEAR] = res
        print(f"\n  {YEAR} done in {time.time()-t0:.0f}s")

print(f"\n{'#'*80}")
print("  Buffer sweep complete.")
print(f"{'#'*80}")


### 7. AUPRC Random Baseline

For a random classifier, AUPRC equals the **fraction of positive (error) pixels** in the evaluation region. This cell computes that prevalence within the FCER ($r_d$ = ASD).


In [ ]:
from scipy.ndimage import binary_dilation

ASD_RADIUS_PX = 4   # dilation radius for baseline computation

_struct = _disk(ASD_RADIUS_PX)

for ARCH in ARCHS:
    print(f"\n{'#'*64}")
    print(f"  AUPRC random baseline — {ARCH}  |  r={ASD_RADIUS_PX}px dilation")
    print(f"{'#'*64}")
    print(f"  {'Year':<10}  {'n_patches':>10}  {'n_err_px':>12}  {'n_ok_px':>12}  {'prevalence':>12}")
    print(f"  {'-'*62}")

    grand_err = 0
    grand_ok  = 0

    for YEAR in TEST_YEARS:
        npz_path = OUTPUT_DIR / f"eval_{ARCH}_year{YEAR}.npz"
        if not npz_path.exists():
            print(f"  {YEAR:<10}  [missing npz, skipping]")
            continue

        data  = np.load(str(npz_path), allow_pickle=True)
        probs = data["prob"]
        ys    = data["y"]

        year_err  = 0
        year_ok   = 0
        n_patches = 0

        for i in range(len(ys)):
            gt = ys[i].astype(bool)
            if not gt.any():
                continue

            mask  = binary_dilation(gt, structure=_struct)
            flat  = mask.ravel()
            p_bb  = probs[i].ravel()[flat].clip(1e-7, 1 - 1e-7)
            y_buf = ys[i].ravel()[flat].astype(np.int32)
            err   = ((p_bb > 0.5).astype(np.int32) != y_buf)

            if err.sum() == 0 or (~err).sum() == 0:
                continue

            year_err  += int(err.sum())
            year_ok   += int((~err).sum())
            n_patches += 1

        prev = year_err / (year_err + year_ok) if (year_err + year_ok) > 0 else float("nan")
        print(f"  {YEAR:<10}  {n_patches:>10}  {year_err:>12,}  {year_ok:>12,}  {prev:>12.4f}")
        grand_err += year_err
        grand_ok  += year_ok

    grand_prev = grand_err / (grand_err + grand_ok) if (grand_err + grand_ok) > 0 else float("nan")
    print(f"  {'-'*62}")
    print(f"  {'ALL YEARS':<10}  {'':>10}  {grand_err:>12,}  {grand_ok:>12,}  {grand_prev:>12.4f}")
    print(f"\n  Random AUPRC baseline  ≈ {grand_prev:.4f}  "
          f"({grand_prev*100:.2f}% of buffer pixels are errors)")
    print(f"  Random AUROC baseline  = 0.5000  (always, by definition)")

### 8. Wilcoxon Signed-Rank Test

Paired test where each observation is one fire event. Metrics are computed within the FCER ($r_d$ = ASD).


In [ ]:
from scipy.ndimage import binary_dilation
from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score, average_precision_score

ASD_RADIUS_PX = 4   # dilation radius for per-fire AUROC/AUPRC computation

_w_struct = _disk(ASD_RADIUS_PX)


def _per_fire_scores(npz_path):
    """Return (auroc_dudes, auroc_ens, auprc_dudes, auprc_ens) arrays, one entry per fire."""
    data      = np.load(str(npz_path), allow_pickle=True)
    probs     = data["prob"]
    prob_ens  = data["prob_ensemble"]
    unc_dudes = data["unc_dudes"]
    unc_ens   = data["unc_ensemble"]
    ys        = data["y"]

    ad_list, ae_list, pd_list, pe_list = [], [], [], []

    for i in range(len(ys)):
        gt = ys[i].astype(bool)
        if not gt.any():
            continue

        mask  = binary_dilation(gt, structure=_w_struct)
        flat  = mask.ravel()
        p_bb  = probs[i].ravel()[flat].clip(1e-7, 1 - 1e-7)
        u_d   = unc_dudes[i].ravel()[flat]
        u_e   = unc_ens[i].ravel()[flat]
        y_buf = ys[i].ravel()[flat].astype(np.int32)
        err   = ((p_bb > 0.5).astype(np.int32) != y_buf)

        if err.sum() == 0 or (~err).sum() == 0:
            continue

        ad_list.append(roc_auc_score(err, u_d))
        ae_list.append(roc_auc_score(err, u_e))
        pd_list.append(average_precision_score(err, u_d))
        pe_list.append(average_precision_score(err, u_e))

    return (np.array(ad_list), np.array(ae_list),
            np.array(pd_list), np.array(pe_list))


def _wilcoxon_report(scores_d, scores_e, metric_name, year_label):
    n     = len(scores_d)
    delta = scores_d - scores_e

    if n < 10:
        print(f"  {year_label:<10}  {metric_name:<6}  n={n:<5}  "
              f"med_delta={np.median(delta):+.4f}  [too few samples]")
        return

    stat, p = wilcoxon(scores_d, scores_e, alternative="greater", zero_method="wilcox")

    W_plus  = stat
    W_total = n * (n + 1) / 2.0
    r_eff   = (W_plus - (W_total - W_plus)) / W_total

    mag = ("large"  if abs(r_eff) >= 0.5 else "medium" if abs(r_eff) >= 0.3
           else "small" if abs(r_eff) >= 0.1 else "negligible")
    sig = ("***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s.")
    p_str = f"{p:.2e}" if p < 0.0001 else f"{p:.4f}"

    print(f"  {year_label:<10}  {metric_name:<6}  "
          f"n={n:<5}  "
          f"med_DUDES={np.median(scores_d):.4f}  "
          f"med_Ens={np.median(scores_e):.4f}  "
          f"med_delta={np.median(delta):+.4f}  "
          f"p={p_str} {sig:<4}  r={r_eff:+.3f} ({mag})")


for ARCH in ARCHS:
    print(f"\n{'='*115}")
    print(f"  Wilcoxon Signed-Rank Test — {ARCH}  |  r={ASD_RADIUS_PX}px buffer")
    print(f"  H1: DUDES > Ensemble (one-sided)")
    print(f"{'='*115}")

    all_ad, all_ae, all_pd, all_pe = [], [], [], []

    for YEAR in TEST_YEARS:
        npz_path = OUTPUT_DIR / f"eval_{ARCH}_year{YEAR}.npz"
        if not npz_path.exists():
            print(f"  {YEAR:<10}  [missing npz, skipping]")
            continue

        ad, ae, pd_, pe = _per_fire_scores(npz_path)
        all_ad.append(ad);  all_ae.append(ae)
        all_pd.append(pd_); all_pe.append(pe)

        _wilcoxon_report(ad,  ae,  "AUROC", str(YEAR))
        _wilcoxon_report(pd_, pe,  "AUPRC", str(YEAR))
        print()

    # ── Pooled across all years ───────────────────────────────
    if all_ad:
        pool_ad = np.concatenate(all_ad);  pool_ae = np.concatenate(all_ae)
        pool_pd = np.concatenate(all_pd);  pool_pe = np.concatenate(all_pe)
        print(f"  {'─'*110}")
        _wilcoxon_report(pool_ad, pool_ae, "AUROC", "ALL YEARS")
        _wilcoxon_report(pool_pd, pool_pe, "AUPRC", "ALL YEARS")

    print(f"\n  Significance: *** p<0.001  ** p<0.01  * p<0.05  n.s. p>=0.05")
    print(f"  Effect size r (rank-biserial): large >=0.5  medium >=0.3  small >=0.1")


### 9. Plot: FCER Sweep

Four figures:

| Figure | Content |
|--------|---------|
| 2×2 AUROC | One panel per test year, shared y-axis |
| 2×2 AUPRC | One panel per test year, shared y-axis |
| 1×2 mean | Mean ± std AUROC and AUPRC across years |
| 1×2 mean | Mean ± std Brier Score and NLL across years |

X-axis: dilation radius $r_d$ in metres (1 px = 375 m).


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

_PX2M = PIXEL_RES_M                                    # 375 m / px

x_numeric        = [r * _PX2M for r in range(MAX_RADIUS_PX + 1)]   # metres
x_tick_positions = [r * _PX2M for r in range(0, MAX_RADIUS_PX + 1, 2)]
x_tick_labels    = [str(int(v)) for v in x_tick_positions]

AUPRC_RAND  = 0.205    # random-classifier AUPRC baseline (from Cell 7)
AUROC_RAND  = 0.5      # random-classifier AUROC baseline (always 0.5)

# Ensemble first (blue, solid) — DUDES second (orange, dashed)
METHODS = {
    "ensemble": {"label": "Ensemble", "color": "#1f77b4", "ls": "-"},
    "dudes":    {"label": "DUDES",    "color": "#ff7f0e", "ls": "--"},
}
METRICS  = ["auroc", "auprc", "brier", "nll"]
ARROWS   = {"auroc": "↑", "auprc": "↑", "brier": "↓", "nll": "↓"}
MTITLES  = {"auroc": "AUROC", "auprc": "AUPRC", "brier": "Brier Score", "nll": "NLL"}

# ── Figure output directory ────────────────────────────────────
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _setup_ax(ax, metric, year=None, legend_loc="best", large_fonts=False):
    fs_tick  = 13 if large_fonts else 10
    fs_label = 15 if large_fonts else 12
    fs_title = 16 if large_fonts else 13
    fs_leg   = 13 if large_fonts else 11

    # ── Horizontal baselines (AUROC / AUPRC only) ─────────────
    if metric == "auroc":
        ax.axhline(y=AUROC_RAND, color="grey", ls=":", lw=2.2,
                   label=f"Baseline = {AUROC_RAND}")
    elif metric == "auprc":
        ax.axhline(y=AUPRC_RAND, color="grey", ls=":", lw=2.2,
                   label=f"Baseline = {AUPRC_RAND}")
    # ── Title ─────────────────────────────────────────────────
    arrow = ARROWS[metric]
    if year is not None:
        ax.set_title(f"{MTITLES[metric]} ({arrow}) [Year {year}]",
                     fontsize=fs_title, fontweight="bold")
    else:
        ax.set_title(f"{MTITLES[metric]} ({arrow})",
                     fontsize=fs_title, fontweight="bold")
    # ── Axes formatting ───────────────────────────────────────
    ax.set_xticks(x_tick_positions)
    ax.set_xticklabels(x_tick_labels, fontsize=fs_tick, fontweight="bold")
    ax.set_xlim(0, MAX_RADIUS_PX * _PX2M)
    ax.set_xlabel("Dilation Radius $r_d$ (m)", fontsize=fs_label, fontweight="bold")
    ax.yaxis.set_major_locator(mticker.MultipleLocator(0.2))
    ax.tick_params(axis="y", labelsize=fs_tick)
    for lbl in ax.get_yticklabels():
        lbl.set_fontweight("bold")
    leg = ax.legend(fontsize=fs_leg, loc=legend_loc)
    if leg:
        for txt in leg.get_texts():
            txt.set_fontweight("bold")
    ax.grid(True, alpha=0.3)


for ARCH in ARCHS:
    if ARCH not in sweep_results:
        print(f"  {ARCH}: no sweep results in memory, skipping.")
        continue

    arch_sweep = sweep_results[ARCH]

    # ── Collect per-year arrays for mean computation ──────────
    all_vals = {m: {met: [] for met in METRICS} for m in METHODS}
    for YEAR in TEST_YEARS:
        if YEAR not in arch_sweep:
            continue
        res = arch_sweep[YEAR]
        for mkey in METHODS:
            for met in METRICS:
                all_vals[mkey][met].append(res[mkey][met][:MAX_RADIUS_PX + 1])

    # ── 2×2 AUROC per year ────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharey=True)
    fig.subplots_adjust(hspace=0.4)
    for ax_idx, YEAR in enumerate(TEST_YEARS):
        ax = axes.flatten()[ax_idx]
        if YEAR not in arch_sweep:
            ax.set_title(f"AUROC (↑) [Year {YEAR}] (missing)", fontsize=13, fontweight="bold"); continue
        for mkey, mstyle in METHODS.items():
            ax.plot(x_numeric, arch_sweep[YEAR][mkey]["auroc"][:MAX_RADIUS_PX + 1],
                    label=mstyle["label"], color=mstyle["color"],
                    linestyle=mstyle["ls"], linewidth=2, marker="o", markersize=2)
        _setup_ax(ax, "auroc", year=YEAR, legend_loc="lower right")
    fig.savefig(FIG_DIR / f"auroc_per_year_{ARCH}.pdf", bbox_inches="tight")
    plt.show()
    print(f"  Saved → {FIG_DIR / f'auroc_per_year_{ARCH}.pdf'}")

    # ── 2×2 AUPRC per year ────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharey=True)
    fig.subplots_adjust(hspace=0.4)
    for ax_idx, YEAR in enumerate(TEST_YEARS):
        ax = axes.flatten()[ax_idx]
        if YEAR not in arch_sweep:
            ax.set_title(f"AUPRC (↑) [Year {YEAR}] (missing)", fontsize=13, fontweight="bold"); continue
        for mkey, mstyle in METHODS.items():
            ax.plot(x_numeric, arch_sweep[YEAR][mkey]["auprc"][:MAX_RADIUS_PX + 1],
                    label=mstyle["label"], color=mstyle["color"],
                    linestyle=mstyle["ls"], linewidth=2, marker="o", markersize=2)
        _setup_ax(ax, "auprc", year=YEAR)
    fig.savefig(FIG_DIR / f"auprc_per_year_{ARCH}.pdf", bbox_inches="tight")
    plt.show()
    print(f"  Saved → {FIG_DIR / f'auprc_per_year_{ARCH}.pdf'}")

    # ── Mean AUROC + AUPRC — 1×2 ─────────────────────────────
    fig, (ax_auroc, ax_auprc) = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
    for mkey, mstyle in METHODS.items():
        for metric, ax in [("auroc", ax_auroc), ("auprc", ax_auprc)]:
            arr  = np.array(all_vals[mkey][metric])
            mean = arr.mean(axis=0)
            std  = arr.std(axis=0)
            ax.plot(x_numeric, mean,
                    label=mstyle["label"], color=mstyle["color"],
                    linestyle=mstyle["ls"], linewidth=2.5, marker="o", markersize=2)
            ax.fill_between(x_numeric, mean - std, mean + std,
                            color=mstyle["color"], alpha=0.15)
    _setup_ax(ax_auroc, "auroc", legend_loc="lower right", large_fonts=True)
    _setup_ax(ax_auprc, "auprc", large_fonts=True)
    plt.tight_layout()
    fig.savefig(FIG_DIR / f"auroc_auprc_mean_{ARCH}.pdf", bbox_inches="tight")
    plt.show()
    print(f"  Saved → {FIG_DIR / f'auroc_auprc_mean_{ARCH}.pdf'}")

    # ── Mean Brier + NLL — 1×2 ───────────────────────────────
    fig, (ax_brier, ax_nll) = plt.subplots(1, 2, figsize=(13, 4.5))
    for mkey, mstyle in METHODS.items():
        for metric, ax in [("brier", ax_brier), ("nll", ax_nll)]:
            arr  = np.array(all_vals[mkey][metric])
            mean = arr.mean(axis=0)
            std  = arr.std(axis=0)
            ax.plot(x_numeric, mean,
                    label=mstyle["label"], color=mstyle["color"],
                    linestyle=mstyle["ls"], linewidth=2.5, marker="o", markersize=2)
            ax.fill_between(x_numeric, mean - std, mean + std,
                            color=mstyle["color"], alpha=0.15)
    _setup_ax(ax_brier, "brier", large_fonts=True)
    _setup_ax(ax_nll, "nll", large_fonts=True)
    plt.tight_layout()
    fig.savefig(FIG_DIR / f"brier_nll_mean_{ARCH}.pdf", bbox_inches="tight")
    plt.show()
    print(f"  Saved → {FIG_DIR / f'brier_nll_mean_{ARCH}.pdf'}")

print(f"\nAll figures saved to: {FIG_DIR}")


### 10. Plot: Qualitative Figure

Selects two representative fire samples  (one small and one large) and renders a **2×4 figure** showing the GT fire mask, FCER, Ensemble uncertainty, and DUDES uncertainty side-by-side.

**Sample selection** is based on per-sample ΔAUROC = AUROC(DUDES) − AUROC(Ensemble) within the FCER ($r_D$ = ASD):

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.font_manager as fm
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from matplotlib.offsetbox import TextArea, HPacker, AnchoredOffsetbox
from scipy.ndimage import binary_dilation as _sp_dilation, label as _sp_label
from sklearn.metrics import roc_auc_score

VIS_YEAR      = 2021
VIS_ARCH      = ARCHS[0]
BUFFER_RADIUS = 4   # px

SMALL_FIRE_MIN = 50
SMALL_FIRE_MAX = 100
BIG_FIRE_MIN   = 400

# ── GT contour color in uncertainty maps ──────────────────────
CONTOUR_DARK_GREEN = "#00ff00"    # neon green  (fire pixels)
CONTOUR_BLUE       = "#4a47ee"    # blue-purple (ASD buffer ring)

CONTOUR_COLOR = CONTOUR_DARK_GREEN   # ← change here

# ── RGB tuples derived from constants (used in panel fills) ───
_rgb_blue  = mcolors.to_rgb(CONTOUR_BLUE)
_rgb_green = mcolors.to_rgb(CONTOUR_DARK_GREEN)

# ── Scale bar ─────────────────────────────────────────────────
SCALEBAR_PX = 8                    # ← change to adjust bar length
SCALEBAR_M  = SCALEBAR_PX * 375    # metres  (8 × 375 = 3000 m)
scalebar_px = SCALEBAR_PX

def _add_scalebar(ax, size_px, label, color="white", fontsize=13):
    fontprops = fm.FontProperties(size=fontsize, weight="bold")
    bar = AnchoredSizeBar(
        ax.transData, size_px, label,
        loc="lower right", pad=0.3, color=color,
        frameon=False, size_vertical=max(1, size_px * 0.15),
        fontproperties=fontprops,
    )
    ax.add_artist(bar)

def _colored_title_col0(ax, fontsize=16):
    """Single-line title: ■ GT Fire Mask (green square)."""
    packed = HPacker(children=[
        TextArea("\u25a0", textprops=dict(fontsize=fontsize + 4, fontweight="bold", color=CONTOUR_DARK_GREEN)),
        TextArea(" GT Fire Mask", textprops=dict(fontsize=fontsize, fontweight="bold", color="black")),
    ], align="center", pad=0, sep=0)
    anch = AnchoredOffsetbox(
        loc="lower center", child=packed, pad=0., frameon=False,
        bbox_to_anchor=(0.5, 1.0), bbox_transform=ax.transAxes, borderpad=0.,
    )
    ax.add_artist(anch)

def _colored_title_col1(ax, fontsize=16):
    """Single-line title: ■ FCER (r_D = ASD) (blue square)."""
    packed = HPacker(children=[
        TextArea("\u25a0", textprops=dict(fontsize=fontsize + 4, fontweight="bold", color=CONTOUR_BLUE)),
        TextArea(" FCER ($r_D$ = ASD)", textprops=dict(fontsize=fontsize, fontweight="bold", color="black")),
    ], align="center", pad=0, sep=0)
    anch = AnchoredOffsetbox(
        loc="lower center", child=packed, pad=0., frameon=False,
        bbox_to_anchor=(0.5, 1.0), bbox_transform=ax.transAxes, borderpad=0.,
    )
    ax.add_artist(anch)

# ── Disk structuring element (r = BUFFER_RADIUS) ─────────────
_r    = np.arange(-BUFFER_RADIUS, BUFFER_RADIUS + 1)
_disk = (_r[:, None]**2 + _r[None, :]**2) <= BUFFER_RADIUS**2

# ── Load npz for this year ────────────────────────────────────
npz_path = OUTPUT_DIR / f"eval_{VIS_ARCH}_year{VIS_YEAR}.npz"
assert npz_path.exists(), f"Missing: {npz_path}. Run Cell 3 first."
data = np.load(str(npz_path), allow_pickle=True)

probs     = data["prob"]
prob_ens  = data["prob_ensemble"]
unc_dudes = data["unc_dudes"]
unc_ens   = data["unc_ensemble"]
ys        = data["y"]

print(f"Loaded {len(ys)} samples  |  arch={VIS_ARCH}  year={VIS_YEAR}")

# ── Compute per-sample ΔAUROC ─────────────────────────────────
records = []

for idx in range(len(ys)):
    gt = ys[idx].astype(bool)

    labeled, n_comp = _sp_label(gt)
    if n_comp == 0:
        continue
    largest_cc   = int(max(np.bincount(labeled.ravel())[1:]))
    n_fire_total = int(gt.sum())

    in_small = (SMALL_FIRE_MIN <= n_fire_total <= SMALL_FIRE_MAX and
                SMALL_FIRE_MIN <= largest_cc    <= SMALL_FIRE_MAX)
    in_big   = largest_cc >= BIG_FIRE_MIN

    if not (in_small or in_big):
        continue

    buf_mask  = _sp_dilation(gt, structure=_disk)
    pred_bb   = probs[idx] > 0.5
    error_map = (pred_bb != gt).astype(int)
    err_buf   = error_map[buf_mask]

    if err_buf.sum() == 0 or (~err_buf.astype(bool)).sum() == 0:
        continue

    unc_d_buf = unc_dudes[idx][buf_mask]
    unc_e_buf = unc_ens[idx][buf_mask]
    auroc_d   = roc_auc_score(err_buf, unc_d_buf)
    auroc_e   = roc_auc_score(err_buf, unc_e_buf)

    records.append({
        "idx":         idx,
        "delta":       auroc_d - auroc_e,
        "auroc_dudes": auroc_d,
        "auroc_ens":   auroc_e,
        "n_fire":      n_fire_total,
        "largest_cc":  largest_cc,
        "in_small":    in_small,
        "in_big":      in_big,
        "gt":          gt,
        "unc_dudes":   unc_dudes[idx],
        "unc_ens":     unc_ens[idx],
        "prob_ens":    prob_ens[idx],
        "buf_mask":    buf_mask,
    })

    if (idx + 1) % 500 == 0:
        print(f"  {idx+1}/{len(ys)}  valid so far: {len(records)}")

print(f"\nTotal valid samples: {len(records)}")

# ── Split into pools ──────────────────────────────────────────
pool_small = [r for r in records if r["in_small"]]
pool_big   = [r for r in records if r["in_big"]]
print(f"Pool small (total & CC in [{SMALL_FIRE_MIN},{SMALL_FIRE_MAX}] px): {len(pool_small)}")
print(f"Pool big   (largest CC ≥ {BIG_FIRE_MIN} px)                    : {len(pool_big)}")

assert pool_small, "No samples in small-fire pool"
assert pool_big,   "No samples in big-fire pool"

# ── Select sample closest to pool mean ΔAUROC ─────────────────
def _pick_mean(pool):
    deltas     = np.array([r["delta"] for r in pool])
    mean_delta = deltas.mean()
    return pool[int(np.argmin(np.abs(deltas - mean_delta)))], mean_delta

sample_small, mean_delta_small = _pick_mean(pool_small)
sample_big,   mean_delta_big   = _pick_mean(pool_big)

for label, samp, mean_d in [
    ("Small fire (typical)", sample_small, mean_delta_small),
    ("Big fire   (typical)", sample_big,   mean_delta_big),
]:
    print(f"\n{label}:")
    print(f"  idx={samp['idx']}  largest_cc={samp['largest_cc']}px  "
          f"total_fire_px={samp['n_fire']}")
    print(f"  ΔAUROC={samp['delta']:.4f}  (pool mean={mean_d:.4f})  "
          f"DUDES={samp['auroc_dudes']:.4f}  Ens={samp['auroc_ens']:.4f}")

# ── Visualisation — 2×4 compact crop ──────────────────────────
CROP_PAD = 20   # px margin around GT bounding box on each side

def _gt_bbox(gt, pad):
    h, w = gt.shape
    rows = np.where(gt.any(axis=1))[0]
    cols = np.where(gt.any(axis=0))[0]
    r0, r1 = rows[0], rows[-1]
    c0, c1 = cols[0], cols[-1]
    cr = (r0 + r1) // 2
    cc = (c0 + c1) // 2
    half = max(r1 - r0, c1 - c0) // 2 + pad
    return (max(0, cr - half), min(h - 1, cr + half),
            max(0, cc - half), min(w - 1, cc + half))

# ── Single shared vmax across all 4 uncertainty crops ─────────
_all_unc = [sample_small["unc_ens"], sample_big["unc_ens"],
            sample_small["unc_dudes"], sample_big["unc_dudes"]]
vmax_unc = float(np.percentile(np.concatenate([u.ravel() for u in _all_unc]), 99))
norm_unc = mcolors.Normalize(vmin=0, vmax=vmax_unc)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))

for row, rec, row_label in [
    (0, sample_small,
     f"Small fire  CC={sample_small['largest_cc']} px\n"
     f"ΔAUROC={sample_small['delta']:.3f}  "
     f"(D={sample_small['auroc_dudes']:.3f}, E={sample_small['auroc_ens']:.3f})"),
    (1, sample_big,
     f"Big fire  CC={sample_big['largest_cc']} px\n"
     f"ΔAUROC={sample_big['delta']:.3f}  "
     f"(D={sample_big['auroc_dudes']:.3f}, E={sample_big['auroc_ens']:.3f})"),
]:
    gt      = rec["gt"]
    unc_ens = rec["unc_ens"]
    unc_d   = rec["unc_dudes"]

    rmin, rmax, cmin, cmax = _gt_bbox(gt, CROP_PAD)
    gt_crop      = gt     [rmin:rmax+1, cmin:cmax+1]
    unc_ens_crop = unc_ens[rmin:rmax+1, cmin:cmax+1]
    unc_d_crop   = unc_d  [rmin:rmax+1, cmin:cmax+1]
    gt_f_crop    = gt_crop.astype(float)

    dil_crop = _sp_dilation(gt_crop, structure=_disk)

    # Panel 0 — full GT mask with crop rectangle (no scale bar: full image)
    ax = axes[row, 0]
    rgb_full = np.zeros((*gt.shape, 3))
    rgb_full[gt] = _rgb_green
    ax.imshow(rgb_full)
    ax.add_patch(plt.Rectangle(
        (cmin - 0.5, rmin - 0.5), cmax - cmin + 1, rmax - rmin + 1,
        linewidth=2.0, edgecolor="white", facecolor="none", linestyle=":",
    ))
    ax.axis("off")
    ax.set_ylabel(row_label, fontsize=15, fontweight="bold", rotation=90,
                  labelpad=6, va="center")

    # Panel 1 — FCER zone fully filled blue + fire boundary as green contour
    ax = axes[row, 1]
    rgb_crop = np.zeros((*gt_crop.shape, 3))
    rgb_crop[dil_crop] = _rgb_blue
    ax.imshow(rgb_crop)
    ax.contour(gt_f_crop, levels=[0.5], colors=CONTOUR_DARK_GREEN, linewidths=2.0)
    _add_scalebar(ax, scalebar_px, f"{SCALEBAR_M // 1000} km")
    ax.axis("off")

    # Panel 2 — Ensemble uncertainty crop + GT contour + scale bar
    ax = axes[row, 2]
    ax.imshow(unc_ens_crop, cmap="plasma", norm=norm_unc)
    ax.contour(gt_f_crop, levels=[0.5], colors=CONTOUR_COLOR, linewidths=2.0)
    _add_scalebar(ax, scalebar_px, f"{SCALEBAR_M // 1000} km")
    ax.axis("off")

    # Panel 3 — DUDES uncertainty crop + GT contour + scale bar
    ax = axes[row, 3]
    ax.imshow(unc_d_crop, cmap="plasma", norm=norm_unc)
    ax.contour(gt_f_crop, levels=[0.5], colors=CONTOUR_COLOR, linewidths=2.0)
    _add_scalebar(ax, scalebar_px, f"{SCALEBAR_M // 1000} km")
    ax.axis("off")

    if row == 0:
        _colored_title_col0(axes[0, 0])
        _colored_title_col1(axes[0, 1])
        axes[0, 2].set_title("Ensemble unc", fontsize=16, fontweight="bold")
        axes[0, 3].set_title("DUDES unc", fontsize=16, fontweight="bold")

plt.tight_layout()
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
sm = cm.ScalarMappable(cmap="plasma", norm=norm_unc)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Uncertainty", fontsize=16, fontweight="bold")
cbar.ax.tick_params(labelsize=15)

fig_path = FIG_DIR / f"qualitative_{VIS_ARCH}_year{VIS_YEAR}.pdf"
fig.savefig(fig_path, bbox_inches="tight")
print(f"Saved → {fig_path}")
plt.show()
